# 3.2 Powers and Correlations: Ensemble of Scatterers

**Learning objectives:**
- Derive composite voltage V_hh from N_s scatterers using double summation
- Distinguish single-sum (permanent) vs double-sum (fluctuating) terms
- Understand ensemble average and its relationship to temporal average
- Master six covariance equations for SHV, HSHV, and VSVH modes


## Theory: Composite Voltage

### Eq. (3.4) — Composite voltage from N_s scatterers

The received voltage from an ensemble of $N_s$ scatterers is the coherent sum of individual scatterer contributions:

$$V_{hh} = B \sum_{i=1}^{N_s} F_i G_i s_{hh}^{(i)} \exp\left[-j 2 \,\mathrm{Re}(k_h) r_i\right]$$

where:
- $B = \sqrt{P_h^t} \, \frac{g \lambda}{4\pi} \, e^{-j \phi_h^{\text{sys}}}$ is the system constant
- $F_i = \frac{1}{r_i^2 \, l_h(r_i)}$ is the range/path loss factor for scatterer $i$
- $G_i = f^2(\theta_i, \phi_i) W_i$ is the antenna gain factor (pattern $\times$ weighting)

### Eq. (3.5a) — Horizontal channel voltage

$$V_{hh} = B \sum_{i=1}^{N_s} F_i G_i s_{hh}^{(i)} e^{-j 2 \mathrm{Re}(k_h) r_i}$$

### Eq. (3.5b) — Vertical channel voltage

$$V_{vv} = C \sum_{i=1}^{N_s} F_i G_i s_{vv}^{(i)} e^{-j 2 \mathrm{Re}(k_v) r_i}$$

with $C = \sqrt{P_v^t} \, \frac{g \lambda}{4\pi} \, e^{-j \phi_v^{\text{sys}}}$.

The radar measures power by squaring the magnitude of these complex voltages.


## Theory: Double Summation

### Eq. (3.6) — Power from double summation

The received power is proportional to $|V_{hh}|^2$, which produces a **double summation** over all scatterer pairs:

$$|V_{hh}|^2 = B B^* \sum_{i=1}^{N_s} \sum_{k=1}^{N_s} F_i F_k G_i G_k s_{hh}^{(i)} (s_{hh}^{(k)})^* \, e^{-j 2 \mathrm{Re}(k_h)(r_i - r_k)}$$

**First term — Single sum ($i = k$):**
When $i = k$, the phase term disappears and we get:
$$|V_{hh}|^2_{i=k} = |B|^2 \sum_{i=1}^{N_s} F_i^2 G_i^2 |s_{hh}^{(i)}|^2$$

This term carries **permanent scatterer property information** (shape, size, dielectric constant). It is independent of scatterer positions and does not fluctuate with time.

**Second term — Double sum ($i \neq k$):**
When $i \neq k$, the phase $e^{-j 2 \mathrm{Re}(k_h)(r_i - r_k)}$ depends on the pairwise range differences. This term **fluctuates with scatterer positions** as particles move. Over time, temporal averaging drives this term toward zero.

The separation of these two terms is fundamental to understanding radar echo statistics from weather echoes.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def plot_power_distribution(N_s=50, seed=42):
    """Show exponential power distribution emerging as N_s increases."""
    np.random.seed(seed)
    # Simulate N_s scatterers with random phases
    M = 500  # number of trials

    powers = []
    for _ in range(M):
        phases = np.random.uniform(0, 2*np.pi, N_s)
        V = np.sum(np.exp(1j * phases))
        powers.append(np.abs(V)**2 / N_s)

    powers = np.array(powers)
    mean_p = np.mean(powers)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram with exponential fit
    ax1.hist(powers, bins=50, density=True, alpha=0.7, color='steelblue')
    x_exp = np.linspace(0, max(powers)*1.5, 200)
    ax1.plot(x_exp, (1/mean_p) * np.exp(-x_exp/mean_p), 'r-', linewidth=2,
             label=f'Exponential (μ={mean_p:.2f})')
    ax1.set_xlabel('Normalized Power')
    ax1.set_ylabel('Probability Density')
    ax1.set_title(f'Power Distribution (N_s={N_s}, M={M} trials)')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # Variance vs N_s
    Ns_range = np.arange(5, 201, 10)
    variances = []
    for N in Ns_range:
        trial_powers = []
        for _ in range(80):
            phases = np.random.uniform(0, 2*np.pi, int(N))
            V = np.sum(np.exp(1j * phases))
            trial_powers.append(np.abs(V)**2 / N)
        variances.append(np.var(trial_powers))

    ax2.plot(Ns_range, variances, 'b-', linewidth=2)
    ax2.set_xlabel('Number of Scatterers N_s')
    ax2.set_ylabel('Variance of Power Estimate')
    ax2.set_title('Variance Reduction with More Scatterers')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(plot_power_distribution,
         N_s=IntSlider(value=50, min=10, max=500, step=10, description=r'N_s:'),
         seed=IntSlider(value=42, min=1, max=100, step=1, description='Seed:'))

interactive(children=(IntSlider(value=50, description='N_s:', max=500, min=10, step=10), IntSlider(value=42, d…

<function __main__.plot_power_distribution(N_s=50, seed=42)>

## Theory: Ensemble Average and Weather Radar Equation

### Eq. (3.7a/b/c) — Temporal averaging definition

Power is defined via temporal averaging over $M$ samples:
$$P_{hh} = \frac{1}{M} \sum_{m=1}^{M} |V_{hh,m}|^2$$

$$P_{vv} = \frac{1}{M} \sum_{m=1}^{M} |V_{vv,m}|^2$$

$$P_{hv} = \frac{1}{M} \sum_{m=1}^{M} |V_{hv,m}|^2$$

As $M \to \infty$, the **ensemble average** $\langle P_{hh} \rangle$ equals the time average — this is the **ergodic hypothesis** for weather signals.

### Eq. (3.8) — Ensemble average of power

After ensemble averaging over all possible scatterer configurations, the fluctuating double-sum term vanishes:
$$\langle P_{hh} \rangle = |B|^2 \int \frac{|W(r)|^2}{r^4 l_h^2(r)} \, |\langle s_{hh}(r) \rangle|^2 dr$$

### Eq. (3.9) — Scatterer property ensemble average

$$\langle |s_{hh}|^2 \rangle = \int N(X) \, |s_{hh}|^2 \, dX$$

where $N(X)$ is the particle distribution function in property space $X$ (size, shape, orientation, dielectric).

When $N_s$ is large (typically $N_s \gtrsim 10$), the distribution of power becomes **exponential** — a key result for weather radar statistics.

### Eq. (3.10) — Full weather radar equation (integral form)

$$P_{hh}(r_0) = \frac{P_h^t g^2 \lambda^2}{2^8 \pi^3 l_h l_r} \int \frac{|W(r - r_0)|^2}{r^2 l_h^2(r)} \langle |s_{hh}|^2 \rangle dr$$

### Eq. (3.11) — Angular integral reduction

The angular integration over the beam yields the beamfilling factor and range-weighting reduction.

### Eq. (3.12) — Range weighting loss

$$l_r = \frac{c \tau / 2}{\int |W(r)|^2 dr}$$

For a rectangular pulse with matched filter, $l_r = 1.5$ (equivalent to 1.76 dB loss).

### Eq. (3.13) — Final weather radar equation

$$P_{hh}(r_0) = C_1 \frac{\pi^3}{2^8 \ln 2} \int \frac{|W(r - r_0)|^2}{r^2 l_h^2(r)} \langle |s_{hh}|^2 \rangle dr$$

with system constant $C_1$ defined below.

### Eq. (3.22) — System constant $C_1$

$$C_1 = \frac{P^t g^2 \lambda^2 c \tau \theta_1^2}{2^8 \pi l_r \ln 2}$$

where $\theta_1$ is the half-power beamwidth.


## Theory: Correlations

### Eq. (3.14a) — Cross-correlation $R_{hv}$

$$R_{hv} = \langle V_{hh}^* \, V_{vv} \rangle = \langle V_{hh} V_{vv}^* \rangle$$

The cross-correlation between H and V channels carries information about particle shape and orientation.

### Eq. (3.14b) — Autocorrelation $R_{hh}(T_s)$

$$R_{hh}(T_s) = \langle V_{hh}^*(t) \, V_{hh}(t + T_s) \rangle$$

This is the **Doppler spectrum** autocorrelation at lag $T_s$ (pulse repetition interval).

### Eq. (3.15) — Ensemble average of cross-correlation with DP terms

$$\langle R_{hv} \rangle = |B|^2 \int \frac{|W(r)|^2}{r^4 l_h(r) l_v(r)} \langle s_{hh}^* s_{vv} \rangle e^{-j(\phi_h^{\text{sys}} - \phi_v^{\text{sys}})} dr$$

The differential phase $\Delta\phi^{\text{sys}} = \phi_h^{\text{sys}} - \phi_v^{\text{sys}}$ is a system property.

### Eq. (3.16) — Autocorrelation expression

$$\langle R_{hh}(T_s) \rangle = |B|^2 \int \frac{|W(r)|^2}{r^4 l_h^2(r)} \langle s_{hh}^* s_{hh}(r, t+T_s) \rangle dr$$

### Eq. (3.17) — Gaussian velocity distribution

The Doppler spectrum width arises from a Gaussian distribution of radial velocities:
$$p(v) = \frac{1}{\sqrt{2\pi} \sigma_v} \exp\left(-\frac{(v - \langle v \rangle)^2}{2\sigma_v^2}\right)$$

### Eq. (3.18) — Autocorrelation with velocity factor

$$\langle R_{hh}(T_s) \rangle = \langle P_{hh} \rangle \, \rho(T_s) \, e^{-j 2 k_h \langle v \rangle T_s}$$

where $\rho(T_s)$ is the normalized autocorrelation from velocity spread.

### Eq. (3.19) — Normalized autocorrelation

$$\rho(T_s) = \exp\left(-\frac{8 \pi^2 \sigma_v^2 T_s^2}{\lambda^2}\right)$$

### Eq. (3.20) — Mean Doppler velocity

$$\langle v \rangle = -\frac{\lambda}{4\pi T_s} \arg\left[\langle R_{hh}(T_s) \rangle\right]$$

### Eq. (3.21) — Spectrum width

$$\sigma_v = -\frac{\lambda}{2\sqrt{2\pi} T_s} \ln\left(\frac{|\langle R_{hh}(T_s) \rangle|}{\langle P_{hh} \rangle}\right)$$


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def plot_doppler_correlation(T_s_us=0.25, sigma_v=2.0, v_mean=0.0):
    """Show autocorrelation and spectrum width relationship."""
    lambda_ = 0.1  # 10 cm wavelength (S-band)
    T_s = T_s_us * 1e-3  # pulse repetition time in ms
    sigma_v_ms = sigma_v  # m/s

    # Compute rho(T_s) from Eq. (3.19)
    rho = np.exp(-8 * np.pi**2 * sigma_v_ms**2 * T_s**2 / lambda_**2)

    # Autocorrelation magnitude (normalized)
    R_hh_norm = rho * np.exp(-4j * np.pi * v_mean * T_s / lambda_)

    # Range of spectrum widths
    sigma_range = np.linspace(0.1, 8, 200)
    rho_range = np.exp(-8 * np.pi**2 * sigma_range**2 * T_s**2 / lambda_**2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # rho vs spectrum width
    ax1.plot(sigma_range, rho_range, 'b-', linewidth=2)
    ax1.axhline(y=rho, color='r', linestyle='--', xmin=0, xmax=1,
                label=f'σ_v = {sigma_v} m/s → ρ = {rho:.3f}')
    ax1.set_xlabel(r'Spectrum Width σ_v (m/s)')
    ax1.set_ylabel(r'ρ(T_s)')
    ax1.set_title('Normalized Autocorrelation vs Spectrum Width')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1.05)

    # Phase for mean velocity
    phase = -4 * np.pi * v_mean * T_s / lambda_
    ax2.bar(['Mean Velocity'], [v_mean], color='steelblue', width=0.4)
    ax2.set_ylabel('Mean Doppler Velocity (m/s)')
    ax2.set_title(f'Mean Velocity Estimation (phase = {np.degrees(phase):.1f} deg)')
    ax2.grid(alpha=0.3, axis='y')
    ax2.set_ylim(-20, 20)

    plt.tight_layout()
    plt.show()

interact(plot_doppler_correlation,
         T_s_us=FloatSlider(value=0.25, min=0.1, max=1.0, step=0.05, description=r'T_s (ms):'),
         sigma_v=FloatSlider(value=2.0, min=0.1, max=8.0, step=0.1, description=r'σ_v (m/s):'),
         v_mean=FloatSlider(value=0.0, min=-20, max=20, step=0.5, description=r'<v> (m/s):'))

interactive(children=(FloatSlider(value=0.25, description='T_s (ms):', max=1.0, min=0.1, step=0.05), FloatSlid…

<function __main__.plot_doppler_correlation(T_s_us=0.25, sigma_v=2.0, v_mean=0.0)>

## Theory: SHV/HSHV/VSVH Mode Equations

### Eq. (3.23) — Mean power $\langle P_{hh}(r_0) \rangle$ (SHV mode)

$$\langle P_{hh}(r_0) \rangle = \frac{C_1}{\ln 2} \int \frac{|W(r - r_0)|^2}{r^2 l_h^2(r)} \langle |s_{hh}|^2 \rangle dr$$

### Eq. (3.24) — Mean power $\langle P_{vv}(r_0) \rangle$ (SHV mode)

$$\langle P_{vv}(r_0) \rangle = \frac{C_1}{\ln 2} \int \frac{|W(r - r_0)|^2}{r^2 l_v^2(r)} \langle |s_{vv}|^2 \rangle dr$$

### Eq. (3.25) — Cross-correlation $\langle R_{hv}(r_0) \rangle$ (SHV mode)

$$\langle R_{hv}(r_0) \rangle = \frac{C_1}{\ln 2} e^{-j(\phi_h^{\text{sys}} - \phi_v^{\text{sys}})} \int \frac{|W(r - r_0)|^2}{r^2 l_h(r) l_v(r)} \langle s_{hh}^* s_{vv} \rangle dr$$

where $\Delta\phi^{\text{pro}} = \arg\langle s_{hh}^* s_{vv} \rangle$ is the **backscatter differential phase** and $\Delta\phi^{\text{sys}}$ is the system differential phase.

### Eq. (3.26) — Mean cross-pol power $\langle P_{hv}(r_0) \rangle$ (HSHV/VSVH modes)

$$\langle P_{hv}(r_0) \rangle = \frac{C_1}{\ln 2} \int \frac{|W(r - r_0)|^2}{r^2 l_h(r) l_v(r)} \langle |s_{hv}|^2 \rangle dr$$

### Eq. (3.27) — Cross-correlation $\langle R_{xh}(r_0) \rangle$ (HSHV mode)

$$\langle R_{xh}(r_0) \rangle = \frac{C_1}{\ln 2} e^{-j\phi_v^{\text{sys}}} \int \frac{|W(r - r_0)|^2}{r^2 l_h(r) l_v(r)} \langle s_{vh}^* s_{hh} \rangle dr$$

### Eq. (3.28) — Cross-correlation $\langle R_{xv}(r_0) \rangle$ (VSVH mode)

$$\langle R_{xv}(r_0) \rangle = \frac{C_1}{\ln 2} e^{-j\phi_h^{\text{sys}}} \int \frac{|W(r - r_0)|^2}{r^2 l_h(r) l_v(r)} \langle s_{hv}^* s_{vv} \rangle dr$$

### Eq. (3.42) — Covariance Matrix (upper triangular form)

The backscattering covariance matrix for an ensemble of scatterers with arbitrary orientation:

$$\mathbf{C} = \begin{pmatrix} \langle |S_{hh}|^2 \rangle & \langle S_{hh} S_{vv}^* \rangle & \langle S_{hh} S_{vh}^* \rangle \\ \langle |S_{hv}|^2 \rangle & \langle S_{hv} S_{vv}^* \rangle \\ \langle |S_{vv}|^2 \rangle \end{pmatrix}$$

For weather radar, symmetry reduces the number of independent elements.


In [3]:
from ipywidgets import Dropdown
import matplotlib.pyplot as plt

def plot_mode_comparison(mode='SHV'):
    """Display available measurements per radar mode."""
    modes = {
        'SHV': {
            'powers': [r'$P_{hh}$', r'$P_{vv}$', r'$P_{hv}$'],
            'correlations': [r'$R_{hv}$'],
            'derived': [r'$Z_{hh}$', r'$Z_{DR}$', r'$\rho_{hv}$'],
            'description': 'Simultaneous H+V transmit, simultaneous H+V receive'
        },
        'HSHV': {
            'powers': [r'$P_{hh}$', r'$P_{vv}$', r'$P_{hv}$'],
            'correlations': [r'$R_{hv}$', r'$R_{xh}$'],
            'derived': [r'$Z_{hh}$', r'$Z_{DR}$', r'$\rho_{hv}$', r'$L_{drh}$', r'$\rho_{xh}$'],
            'description': 'H transmit, simultaneous H+V receive'
        },
        'VSVH': {
            'powers': [r'$P_{hh}$', r'$P_{vv}$', r'$P_{hv}$'],
            'correlations': [r'$R_{hv}$', r'$R_{xv}$'],
            'derived': [r'$Z_{hh}$', r'$Z_{DR}$', r'$\rho_{hv}$', r'$\rho_{xv}$'],
            'description': 'V transmit, simultaneous V+H receive'
        }
    }
    m = modes[mode]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axis('off')
    ax.set_title(f'Radar Mode: {mode}')
    text = f"{m['description']}\n\n"
    text += f"Powers ({len(m['powers'])}): {', '.join(m['powers'])}\n\n"
    text += f"Correlations ({len(m['correlations'])}): {', '.join(m['correlations'])}\n\n"
    text += f"Derived Variables ({len(m['derived'])}): {', '.join(m['derived'])}"
    ax.text(0.05, 0.5, text, fontsize=11, va='center', fontfamily='monospace',
            transform=ax.transAxes)
    plt.tight_layout()
    plt.show()

interact(plot_mode_comparison,
         mode=Dropdown(options=['SHV', 'HSHV', 'VSVH'], description='Mode:'))

interactive(children=(Dropdown(description='Mode:', options=('SHV', 'HSHV', 'VSVH'), value='SHV'), Output()), …

<function __main__.plot_mode_comparison(mode='SHV')>

## Summary

Key takeaways from this notebook:

1. **Composite voltage** — coherent sum of $N_s$ scatterer contributions (Eq. 3.4, 3.5a/b)
2. **Double summation** — power contains single-sum (permanent, position-independent) and double-sum (fluctuating, position-dependent) terms (Eq. 3.6)
3. **Exponential power distribution** — emerges when $N_s \gtrsim 10$ scatterers contribute
4. **Ensemble average** — equals temporal average as $M \to \infty$ (ergodic hypothesis)
5. **Weather radar equation** — integral form with range weighting loss $l_r = 1.5$ (Eq. 3.10-3.13)
6. **System constant** $C_1$ — combines transmit power, antenna gain, wavelength, and pulse width (Eq. 3.22)
7. **Cross-correlation** $R_{hv} = \langle V_{hh}^* V_{vv} \rangle$ — carries shape/orientation info (Eq. 3.14a)
8. **Autocorrelation** $R_{hh}(T_s)$ — Doppler velocity and spectrum width from lag structure (Eq. 3.14b, 3.18-3.21)
9. **SHV/HSHV/VSVH modes** — each mode provides different subsets of the full covariance matrix (Eq. 3.23-3.28)
10. **Covariance matrix** — 3×3 Hermitian matrix fully characterizing ensemble scatter (Eq. 3.42)

## References

- Doviak & Zrnic (2006) *Doppler Radar and Weather Observations*, Sect. 4.4, Academic Press
- Ryzhkov & Zrnic (2024) *Radar Polarimetry for Weather Observations*, Ch. 3, Springer
- Ryzhkov et al. (2005) *J. Atmos. Oceanic Technol.*, 22, 976–996
